In [ ]:
%load_ext autoreload
%autoreload 2

from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from scaling_llms.models import GPTConfig, GPTModel
from scaling_llms.utils.training import set_determinism
from scaling_llms.experiments import build_raw_model, build_trainer
from scaling_llms.data import make_tokenized_dataset, make_dataloaders, DataLoaderConfig
from scaling_llms.registries import (
    # make_run_registry, 
    # make_dataset_registry, 
    # MakeRunRegistryConfig,
    # MakeDatasetRegistryConfig,
    # DatasetArtifactsDir,
    DatasetIdentity
)



In [ ]:
DATASET_KWARGS = dict(
    dataset_name="Salesforce/wikitext",
    dataset_config="wikitext-103-raw-v1",
    train_split="train[:1%]",  # use 1% of the training data for the proxy
    eval_split="validation[:1%]",  # use 1% of the validation data for the proxy
    tokenizer_name="gpt2_tiktoken",
    text_field="text"
)

CONSTANT_GPT_HPARAMS = dict(
    # --- Dimensions ---
    # n_embd TO DEFINE IN WIDTH-SWEEP BELOW
    n_layer=6,           # depth fixed throughout
    n_head=4,
    # --- MLP ---
    mlp_type="standard_gelu",
    mlp_bias=False,
    d_ff = None,  # will be set to 4*n_embd in GPTModel if None
    # --- Normalization ---
    norm_type="rmsnorm",
    # --- Positional Encoding ---
    pos_encoding_type="rotary",
    # --- Attention / LM head ---
    attn_bias=False,
    tied_embeddings=False,
    lm_head_bias=False,
   # --- Dropout ---
    attn_pdrop=0.0,
    resid_pdrop=0.0,
)

BASE_WIDTH = 128  # your proxy width
WIDTHS = [BASE_WIDTH, 256, 512, 1024, 2048]  # proxy + 4 transfer targets
LOG2LRS = np.linspace(-13, -1, 8, endpoint=False)

In [ ]:
# Make tokenized dataset 
# NOTE: dataset_info -> (dtype, vocab size, etc.)
dataset_dir, dataset_info = make_tokenized_dataset(
    dataset_id=DatasetIdentity(**DATASET_KWARGS),
    dataset_registry=None,
    local_train_mmap_path=Path("/tmp/train.bin"),
    local_eval_mmap_path=None, #Path("/tmp/eval.bin"),
    local_hf_cache_dir=Path("/tmp/hf_cache")
)

Found existing token memmap at /tmp/train3.bin, skipping tokenization.
Found existing token memmap at /tmp/eval3.bin, skipping tokenization.
TokenizedDatasetInfo(vocab_size=50257, eos_id=50256, dtype='uint16', total_train_tokens=1180114, total_eval_tokens=1774)


In [ ]:
# Make dataloaders
dl_config = DataLoaderConfig(
    seq_len=512,
    train_batch_size=8,  # for coord check only
    eval_batch_size=8,
    start_sample_idx=0,
    seed=42              # outer seed; inner seed swept below
)
dls = make_dataloaders(
    local_train_mmap_path=dataset_dir.train_bin,
    local_eval_mmap_path=None,
    dtype=np.dtype(dataset_info.dtype),
    dataloader_config=dl_config
) 
train_dl = dls["train"]

In [ ]:
for batch in train_dl:
    x, y = batch
    print("x:", x.shape, x.dtype)
    print("y:", y.shape, y.dtype)
    break

x: torch.Size([8, 512]) torch.int64
y: torch.Size([8, 512]) torch.int64


# A. Coordinate Check

We fix a given width and initialize multiple models using $N$ different random seeds.  
Each model is trained for a small number of steps. At every training step, we record the activations of every layer:

$$
(\text{width},\ \text{step},\ \text{layer}) \;\rightarrow\; [a_1, a_2, \ldots, a_N].
$$

Then, for each such collection we compute the absolute mean across seeds:

$$(\text{width},\ \text{step},\ \text{layer}) \;\rightarrow\; \operatorname{mean\_abs\_act}
= \frac{1}{N}\sum_{i=1}^N |a_i|.$$

Finally, for every step and every layer, we plot  
$$
\operatorname{mean\_abs\_act} \text{vs.} \text{width},
$$  
and generate these plots for both **SP** and **μP** to compare their behaviors.


**Note:** It suffices to use a small subset of the training dataset for this experiment.


---

SP:
- Final layer's weights receive updates that scale as O(1/sqrt(n))
- Thus, after init logits are expected to be unstable with increasing width
- Mean Abs of Logits for higher widths is expected to diverge as training progresses

muP:
- Final layer's weights receive O(1) updates
- Mean Abs of Logits is exected to be roughly constant for across widths and training steps.

In [ ]:
def collect_activations(
    model: GPTModel,
    idx: torch.Tensor,
) -> dict[str, torch.Tensor]:
    COORD_CHECK_LAYERS = {
        "transformer.wte",
        "transformer.h.0.attn",
        "transformer.h.0.mlp",
        "transformer.h.0",
        "transformer.h.2",
        "transformer.norm_f",
    }

    acts: dict[str, torch.Tensor] = {}
    hooks = []

    def _unwrap(out):
        return out[0] if isinstance(out, tuple) else out

    for name, module in model.named_modules():
        if name not in COORD_CHECK_LAYERS:
            continue

        def _hook(m, inp, out, n=name):
            out = _unwrap(out)
            acts[n] = out.detach().abs().mean()

        hooks.append(module.register_forward_hook(_hook))

    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            x = model._encode(idx)
            logits = model._decode(x)
            acts["logits"] = logits.detach().abs().mean()
    finally:
        for h in hooks:
            h.remove()
        model.train(was_training)

    return acts


def run_coord_check(
    x,
    y,
    widths, 
    seeds,
    use_mup,
    seq_len,
    vocab_size,
    lr,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    x = x.to(device)
    y = y.to(device)

    step2records = defaultdict(list)
    for width in widths:
        layer_name_step2act_list = defaultdict(list)
        for seed in seeds:
            set_determinism(seed)
            # Init model and optimizer
            gpt_hparams = dict(
                n_embd=width,
                mup_base_width=BASE_WIDTH if use_mup else None,
                **CONSTANT_GPT_HPARAMS
            )
            
            model = build_raw_model(
                seq_len=seq_len,
                vocab_size=vocab_size, 
                gpt_hparams=gpt_hparams,
            )
            model.to(device)
            params = model.get_param_groups(base_lr=lr, weight_decay=0.0)
            optimizer = torch.optim.AdamW(params, betas=(0.9, 0.95))

            # Train for a few steps and collect activations
            for step in range(5):
                model.train()
                out = model(x, y)

                acts = collect_activations(model, x)
                for layer_name,act in acts.items():
                    layer_name_step2act_list[(layer_name, step)].append(act)

                optimizer.zero_grad()
                out.loss.backward()
                optimizer.step()

        # Compute mean_abs_act and save records
        for (layer_name, step),act_list in layer_name_step2act_list.items():
            step2records[step].append(dict(
                width=width,
                layer_name=layer_name,
                mean_abs_act=torch.stack(act_list).mean().item()
            ))

    return step2records

def plot_sp_mup_rows(step2records_sp, step2records_mup):

    steps = list(step2records_sp.keys())
    n_cols = len(steps)

    dfs_sp  = [pd.DataFrame(d) for d in step2records_sp.values()]
    dfs_mup = [pd.DataFrame(d) for d in step2records_mup.values()]

    for df in dfs_sp + dfs_mup:
        df["log2_width"] = np.log2(df["width"])

    # ── figure layout ────────────────────────────────────────
    fig, axes = plt.subplots(
        2, n_cols,
        figsize=(5 * n_cols, 8),
        sharey="row",           # share y within each row
        sharex=True,
    )
    # ensure axes is always 2-D
    if n_cols == 1:
        axes = axes.reshape(2, 1)

    row_labels = ["SP", "μP"]
    all_dfs    = [dfs_sp, dfs_mup]

    # ── colour map: one colour per layer, consistent across panels ──
    all_layers = pd.concat(dfs_sp + dfs_mup)["layer_name"].unique()
    cmap       = plt.get_cmap("tab10")
    colour     = {layer: cmap(i % 10) for i, layer in enumerate(all_layers)}

    # ── plot ─────────────────────────────────────────────────
    for row_idx, (row_dfs, row_label) in enumerate(zip(all_dfs, row_labels)):
        for col_idx, (df, step) in enumerate(zip(row_dfs, steps)):
            ax = axes[row_idx, col_idx]

            for layer, grp in df.groupby("layer_name"):
                grp = grp.sort_values("log2_width")
                ax.plot(
                    grp["log2_width"],
                    grp["mean_abs_act"],
                    marker="o",
                    markersize=4,
                    linewidth=1.5,
                    color=colour[layer],
                    label=layer if (row_idx == 0 and col_idx == 0) else None,
                )

            # column title on top row only
            if row_idx == 0:
                ax.set_title(f"Step {step}", fontsize=11, fontweight="bold")

            # row label on leftmost column
            if col_idx == 0:
                ax.set_ylabel(f"{row_label}\nmean |act|", fontsize=10)

            # x-axis ticks: 2^k labels
            tick_vals = list(range(
                int(df["log2_width"].min()),
                int(df["log2_width"].max()) + 1
            ))
            ax.set_xticks(tick_vals)
            ax.set_xticklabels([f"$2^{{{k}}}$" for k in tick_vals], fontsize=9)
            ax.set_xlabel("width", fontsize=9)
            ax.grid(True, linestyle="--", alpha=0.4)
            ax.spines[["top", "right"]].set_visible(False)

    # ── legend (top-right of first subplot) ──────────────────
    axes[0, 0].legend(
        title="layer",
        fontsize=8,
        title_fontsize=9,
        loc="upper right",
        framealpha=0.8,
        edgecolor="0.8",
    )

    fig.suptitle(
        "SP vs μP: mean |activation| vs width across steps",
        fontsize=13,
        fontweight="bold",
        y=1.01,
    )
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
kwargs = dict(
    x=x,
    y=y,
    widths=WIDTHS,
    seeds=[1, 12, 123, 1234, 12345],
    seq_len=dl_config.seq_len,
    vocab_size=dataset_info.vocab_size,
    lr=1e-3,
)

step2records_sp = run_coord_check(**kwargs, use_mup=False)
step2records_mup = run_coord_check(**kwargs, use_mup=True)
fig = plot_sp_mup_rows(step2records_sp, step2records_mup)


In [ ]:
# # Pseudocode for what you're launching
# for width in WIDTHS:
#     for lr in LR_SWEEP:
#         cfg = GPTConfig(
#             n_embd=width,
#             mup_base_width=128,  # always the proxy width
#             n_head=4,
#             n_layer=4,
#             ...
#         )
#         model = GPTModel(cfg)
#         groups = model.get_param_groups(base_lr=lr, weight_decay=0.1)
#         optimizer = torch.optim.AdamW(groups)
#         train(model, optimizer, tokens=100_000_000)
#         log(width=width, lr=lr, val_bpb=evaluate(model))